# MNIST CNN Pure Inference on PYNQ-Z2
Run this notebook **on the PYNQ-Z2 board** (Jupyter at 192.168.2.99)

Requirements:
- `cnn.bit` + `cnn.hwh` in the same directory as this notebook
- `fpga_weights/` folder (w_conv1.npy, b_conv1.npy, etc.) in the same directory
- An image file named `test_image.png` (e.g. a hand-drawn digit)


In [ ]:
import numpy as np
import time
from PIL import Image
from pynq import Overlay, allocate
import pynq.lib.dma


In [ ]:
# Load the FPGA overlay (bitstream + hardware handoff)
print('Loading overlay...')
ol = Overlay('./cnn.bit')
print('Overlay loaded.')
dma = ol.axi_dma_0
cnn = ol.cnn_top_0


In [ ]:
import base64, io
b64_dict = {'w_conv1': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGkyJywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDgsIDEsIDMsIDMpLCB9ICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAp9AJD/ugCK/3IBggCXASwA7v+GAKoAtwINAKf/dwIC/Rv/QgGj/mT+WP48/8r+0QB9/nn/4AHt/2/+KP0PAcMBKf+IABQCOgCv/yYBE/8uAZAAQQEzAUQCbwHPAM4BTf9xANcBj/+8AfoAMP8h/b7+CwDP/zP/QAFYArEABAID/pD/Bv8Q/u/9IP8vAPP+sP0=', 'b_conv1': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGkyJywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDgsKSwgfSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAp+ARYA+P+cAEcBFf/K/9QB', 'w_conv2': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGkyJywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE2LCA4LCAzLCAzKSwgfSAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAoeADcAO/87/6//2//K/or/BwCiAH//2v4SAdAAuf+m/8IAaAC+/6X/jv9R/6/+dv4jAH7/wf6CAHcBxAC//4ABRgFb/9j/cgB6ADkAQP+I/7AAtgB//lf/WABeABAAuP8T/1IALwDT/vv/AwAmAMj/RAAx/9j+8ADW/2T/sf+R/3v/YgB9AC7/6/9sAEwAO/9iAPD/dADX/6EAfQCy/8f/JP/Q/03/hP9BAIUAzABqAEAALv+g/8b/1gAd/zX/e/8CANP/AAFWAAoBpwCx/xoA4P4CAGf/zP4HABcAMwH5/8kA1P+b/wL/Lf+pALn/dwAeAGUAggC0/7P/Kv/W/5EAIgEj/08A3P/4/S3+6//x/8j/qf/k/2D/3f+yAFMAGQBOAFQAkP+W/3IAwgCLAFEA6P9+AOL/awBa/8r/jQDg/zcAnf8p/0L/ef+I//j/yf8VAKP/Kf8FAOD/ev/Y/wEAEAAbAKX/uP6x/87/+v9pAL4AkwCfAHf/Ev8PABcAIwDc/ywAfADH/ywA/P+vABr/ev9PAB4AcACuAEv/Jv+e/+v/BACG/37/Cv/T/3f/9f8SAOf/+f8YAFIAagDn/5IAZAAvACUAiP91AAEAdP/7/5r/PQAm/yb/df98/xf/1v8uAHP//f5pAMv//v/B/+P/CwCB/8D/SQAvAMb/UQDr/9j/LAAnAO//SgA5ALD/IgCO/ywAaABP/1IAmABsABAArP+E/73/zv8rABYAof99/83/2/98/wkAZP9r/zf/f/+y/xYA3f+Q/yIA6f8JADoAYgCyAN0AEgA1AIoAo/8W/30AkgASALj/DgDI/47/bv85/+3+R/98AMAAJgA8ABUBCgEtABQAPAHC/yoAu/84/4j/RgD//2H/PAArAKEAGwDV/8b/1P8tANz/aQDf/8v/1/9KAFUAZ/////H//v91/9D/iP+v/xX/Qv93/8D/Ff9KABsAov/9/+z/2/9U/30AywBhABYAJf8sAYEAo/6FAO0AJwDT/z8AQQD2/mr/KgBl/0H+Lv8sAHIAHgBQ/9sACAEh/34ApwC3/wAAjf9s/0oAj/+X/68ASQBOAFUAp/+Q/2wAzf+e/0kATgCf/0oARP/y/7X/n/+b/8//BgAl/+H/JgAX/7L+EgACAO/+5v5T/5H/egBb/z//RAB4/20AYwB1/44AegAT/+H/twCz/1gADQA+/xEAKQAs/x4AlP+h/2sAhP98/zj+o/+//zP/v/8b/z//d/9U/+T/QADH/4X/RwB0/zoAwADM/3b/0gCJ/xL/4wBP/xMAXAB3/8T/AgDP/4MA0v+3/5AAgQADANz/8P83ADsAnP/k/zwAsf9+/5D/XAD5/5YAWgDTAGkAZv87AND/EABq/9P/3P8eAJD/6v5IAH4AzACOAPAAfP9UAB3/k/8jAKAAVgBfAJn/gP///xj/Mv8w/8P/MAAeAF8AGgCCAFn/T/8TABr/+/88/8QAygD5/z4AjP82/1AAmQFHAXUB5v9MALr/n/+O/9r/Rv+X/xP//P9G/wD/GwBxABMA6v9uAHcALADq/zgAXwB+/5b/BgAlALv/QADKALIAJQAeAO//pP/z/jL/KABW/7L+fv+y/wQAfQDMAAcAiQDv/9gAwv/G/+7/jv+aAOv/LwB9/ywAtf8ZAJz/9f9tAAIANADr/yEAFAAuANb/RgDm/2YAzADd/mIAkgC5/3//2f8M/6n/8//a/5X/zP8h/8H/UADj/lgAr//A/3wArgDw/9H/sv8xABsAzv/KAEEAyf6Z/xQAgACHALMAjwBQAKAA3v9N/0f/aQBK/2n/cADe/uj/zQAU/3P/PwC7/hwARgCr/wsAbgDw/ygASADk/oIAtAAC/6AAogBh/1sAUwDr/0IAav/V/y0AwP4sAD4Akv9iAGn/lv+5/wIAAwDW/5z/BQA1/2AAiQAxADUALwAaACsAOQAnAHYAL/8EAAcA3P5DAFoAOQCl//7/EwBEALf/+/6I/gH/rv+g/wH/FwA5//L/UP8A/2IASAAFAL3/WgAYALEAMwBZ/5UAmgDy/r0AugC7/8oADgBJAAkAeP9CAFgAg/+nAGAAqP+t/6b/0v8JAIT/Jf+1/9v+2f+n/77/if9MAFEAMAAeAB0AyP+b/nT/zP6N/+L/tv+2AJMA9gCnALz/JADf/xAAaQDe/1D/h/9AAIIA0f/wAOEA2wA6AOT/FQAz/wX/NP9nAEQA2/+h/0oALwAE/+/+zv6bANj/9P8xAJj/bQC2AEUA8f9DAZ4AcQBA/2T/fv/a/zwA5f9TAJv/FwCQ/9z/PABc/7b/bwBg/1P/hf+8AC8AVADNAIoB4ABg/uX+p/4JAE0ASQDH/9L/P/8IAMD/uP/j/7r/DACg/5X/wf5vABMACwDGACsA5QAz/+n/Yv9XAGr/zf/zAMMA2gD5/iwA8/8g/+T+E/8BAMkACwCe/sP+2f6qAMYAXwB1AG4AKgC7/7j/XwBpAFMAKQDE/73/uP+WABgAu/8VAB0A7P/v/xAAFAAMAHQAQwBYANP/zv8iAAwAEQCj/zr/IgDy/kL/WP/y/i3/Jv/xAC0AVQAvACMApP+fADEASQDo/5n/pv/B/93/5f9BAGYAVQAZAMX/iP+2/xMA7f/G/xkAtP/ZAB0AYf+M/18AkQCGAD0AdgCL/3T/MABc/1EAQwDF/1j/J/+K/0AANABSAG4ANf8aAGYAI/+YAKcAwf75ANf/Hf+qANz/yv7LAOT/X/+nAFT/UwBbAEj/5v/e/sr/vv88/rwAoP8K/6P/4//J/2QA6P+E/xkAPf+WAKMAHP+w/+MA1f+0/0EBMgCTALkAnf92APD/jP6eABf/fP7QAO3+7/5vAJ3/9v/E/zv/UADt/vX+UgANAAgA4/88AIv/qgA1ADIAjACS/wL/7P/d/6n/q/9j/4P/Pv8tAJMAFAF1/3cA+gDmAHcA3P/WAK3/wf9EACn/N//u/4D/u/9xAGj/DACn//H/ZgDj/04AXQDNAKX/q//LAHf/VwBN/0sANgCi/6X/FAAhABQAVgBiAKUA7v/o/+L/8v9u/5n/1/9Y/4z/EP8=', 'b_conv2': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGkyJywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDE2LCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAqu/9D/DgCq/63/AgBGAB4A4f+k/zYA7v9DAEYA1f/K/w==', 'w_fc': '...', 'b_fc': 'k05VTVBZAQB2AHsnZGVzY3InOiAnPGkyJywgJ2ZvcnRyYW5fb3JkZXInOiBGYWxzZSwgJ3NoYXBlJzogKDEwLCksIH0gICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIAomAAgABQDJ/97/DQDl/9r/zf8HAA=='}
bufs = {}
for name, b64str in b64_dict.items():
    arr = np.load(io.BytesIO(base64.b64decode(b64str)))
    buf = allocate(shape=arr.shape, dtype=np.int16)
    buf[:] = arr
    bufs[name] = buf
SCALE = 1024
REG = { 'w_conv1': 0x10, 'b_conv1': 0x1c, 'w_conv2': 0x28, 'b_conv2': 0x34, 'w_fc': 0x40, 'b_fc': 0x4c }
for name in ['w_conv1', 'b_conv1', 'w_conv2', 'b_conv2', 'w_fc', 'b_fc']:
    cnn.write(REG[name], bufs[name].physical_address & 0xFFFFFFFF)
print('Weights unpacked from base64 and configured in FPGA registers.')


In [ ]:
def preprocess(img_uint8):
    x = img_uint8.astype(np.float32) / 255.0
    x = (x - 0.1307) / 0.3081
    return np.clip(np.round(x * SCALE), -32768, 32767).astype(np.int16)

in_buf  = allocate(shape=(784,), dtype=np.int16)
out_buf = allocate(shape=(10,),  dtype=np.int16)

def run_fpga_inference(img_array):
    in_buf[:] = preprocess(img_array.reshape(784))
    cnn.write(0x00, 1)
    dma.sendchannel.transfer(in_buf)
    dma.recvchannel.transfer(out_buf)
    dma.sendchannel.wait()
    dma.recvchannel.wait()
    return out_buf[:].astype(np.float32) / SCALE


In [ ]:
# Load your own test image (e.g. upload 'test_image.png' to Jupyter)
try:
    img = Image.open('test_image.png').convert('L').resize((28, 28))
    img_array = np.array(img)
    
    # Run Hardware Inference
    logits = run_fpga_inference(img_array)
    pred = np.argmax(logits)
    
    print(pred)
except FileNotFoundError:
    print('Please upload a test_image.png file to the same directory.')


In [ ]:
for name in weight_names:
    bufs[name].freebuffer()
in_buf.freebuffer()
out_buf.freebuffer()
